# AliPOS table-booking discovery

This notebook is read-only. It uses official AliPOS credentials only for authentication, and its safety checks block the deployed restaurant ID from all probes.

In [ ]:
from __future__ import annotations

import json
import os
import re
from pathlib import Path
from typing import Any, Mapping, Optional, Tuple
from urllib.parse import urlsplit


LIVE_FLAG_NAME = "ALLOW_LIVE_ALIPOS_READS"
UUID_PATTERN = r"[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[1-5][0-9a-fA-F]{3}-[89abAB][0-9a-fA-F]{3}-[0-9a-fA-F]{12}"
UUID_RE = re.compile(rf"^{UUID_PATTERN}$")


class ProbeSafetyError(RuntimeError):
    pass


class LiveProbesDisabled(ProbeSafetyError):
    pass


def find_repo_root(start: Optional[Path] = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / ".env").is_file() and (candidate / "notebooks").is_dir():
            return candidate
    raise ProbeSafetyError("Repository root with .env and notebooks was not found")


def read_dotenv(path: Path) -> dict:
    values = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        key = key.strip()
        value = value.strip()
        if len(value) >= 2 and value[0] == value[-1] and value[0] in {"'", '"'}:
            value = value[1:-1]
        values[key] = value
    return values


def _stream_output_text(cell: Mapping[str, Any]) -> str:
    parts = []
    for output in cell.get("outputs", []):
        text = output.get("text", "")
        if isinstance(text, list):
            parts.extend(str(item) for item in text)
        elif text:
            parts.append(str(text))
    return "".join(parts)


def extract_dummy_identifiers(source_notebook: Path) -> Tuple[str, Tuple[str, ...]]:
    document = json.loads(source_notebook.read_text(encoding="utf-8"))
    cells = document.get("cells", [])
    if len(cells) < 10:
        raise ProbeSafetyError("Source notebook does not contain cells 4 and 10")

    config_output = _stream_output_text(cells[3])
    restaurant_match = re.search(rf"Restaurant ID:\s*({UUID_PATTERN})", config_output)
    if restaurant_match is None:
        raise ProbeSafetyError("Dummy restaurant ID was not found in source cell 4")

    created_output = _stream_output_text(cells[9])
    marker = "ID созданных заказов / Yaratilgan buyurtma ID lari:"
    if marker not in created_output:
        raise ProbeSafetyError("Dummy order marker was not found in source cell 10")
    tail = created_output.split(marker, 1)[1].lstrip()
    try:
        order_map, _ = json.JSONDecoder().raw_decode(tail)
    except (TypeError, ValueError) as exc:
        raise ProbeSafetyError("Dummy order map in source cell 10 is invalid") from exc

    order_ids = tuple(order_map.get(name, "") for name in ("cash", "online-order"))
    if any(not UUID_RE.fullmatch(value) for value in order_ids):
        raise ProbeSafetyError("Source cell 10 does not contain two valid dummy order IDs")
    return restaurant_match.group(1), order_ids


def build_probe_config(
    repo_root: Path,
    environ: Optional[Mapping[str, str]] = None,
) -> dict:
    dotenv = read_dotenv(repo_root / ".env")
    process = dict(os.environ if environ is None else environ)

    def configured(name: str) -> str:
        return process.get(name) or dotenv.get(name, "")

    dummy_restaurant_id, dummy_order_ids = extract_dummy_identifiers(
        repo_root / "notebooks" / "alipos_support_report_ru_uz.ipynb"
    )
    return {
        "base_url": configured("ALIPOS_API_BASE_URL").rstrip("/"),
        "client_id": configured("ALIPOS_API_CLIENT_ID"),
        "client_secret": configured("ALIPOS_API_CLIENT_SECRET"),
        "deployed_restaurant_id": configured("ALIPOS_RESTAURANT_ID"),
        "dummy_restaurant_id": dummy_restaurant_id,
        "dummy_order_ids": dummy_order_ids,
        "live_enabled": process.get(LIVE_FLAG_NAME) == "1",
        "timeout_seconds": 15,
        "minimum_interval_seconds": 0.25,
        "max_requests": 120,
    }


def validate_probe_config(
    config: Mapping[str, Any],
    require_live: bool = True,
) -> None:
    required = (
        "base_url",
        "client_id",
        "client_secret",
        "deployed_restaurant_id",
        "dummy_restaurant_id",
    )
    missing = [name for name in required if not str(config.get(name, "")).strip()]
    if missing:
        raise ProbeSafetyError("Required AliPOS configuration is missing")

    parsed = urlsplit(str(config["base_url"]))
    hostname = (parsed.hostname or "").lower()
    if parsed.scheme != "https" or not hostname:
        raise ProbeSafetyError("AliPOS base URL must use HTTPS")
    if hostname != "alipos.uz" and not hostname.endswith(".alipos.uz"):
        raise ProbeSafetyError("Configured host is not an AliPOS host")
    if parsed.username or parsed.password:
        raise ProbeSafetyError("AliPOS base URL must not contain credentials")

    deployed_id = str(config["deployed_restaurant_id"])
    dummy_id = str(config["dummy_restaurant_id"])
    if not UUID_RE.fullmatch(deployed_id) or not UUID_RE.fullmatch(dummy_id):
        raise ProbeSafetyError("Restaurant IDs must be UUIDs")
    if deployed_id.casefold() == dummy_id.casefold():
        raise ProbeSafetyError("Dummy restaurant ID matches the deployed restaurant ID")

    order_ids = tuple(config.get("dummy_order_ids", ()))
    if len(order_ids) != 2 or any(not UUID_RE.fullmatch(str(value)) for value in order_ids):
        raise ProbeSafetyError("Exactly two valid dummy order IDs are required")
    if require_live and not bool(config.get("live_enabled")):
        raise LiveProbesDisabled(f"Set {LIVE_FLAG_NAME}=1 only for the approved live run")
